# CNN Image Classification — CIFAR-10
## Neural networks for energy applications — Day 4, Exercise 1

In this exercise we build and train convolutional neural networks (CNNs) for image
classification on the CIFAR-10 dataset. We start with a simple two-block architecture,
evaluate its performance, diagnose errors, and then change one thing, adding batch
normalization and dropout, to see whether it helps.

**Evaluation protocol.** We hold out 5,000 of the 50,000 training images as a
**validation set**. Validation data are used to monitor training, to keep the best epoch,
to inspect errors and to compare the two models. The 10,000 **test** images are used once,
in Section 8, for the model we have already chosen. Using the test set after every epoch
(or to pick the better model) would turn it into a second validation set.

**Runtime.** To fit a laptop CPU, both models train on the same fixed 20,000-image subset
of the training split for 10 epochs.
This takes about 3–5 minutes on a laptop CPU and 1–2 minutes on a GPU or Apple silicon.
Set `N_TRAIN = 45_000` to use the whole training split; the comparison stays fair as long
as both models get the same budget.

---
## 0 — Imports and Setup

In [ ]:
import copy
import os

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split

import torchvision
import torchvision.transforms as transforms

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Use a GPU if there is one (CUDA, or MPS on Apple silicon), otherwise the CPU.
# GPU results can differ slightly from CPU results even with the same seed.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# CIFAR-10 is downloaded to ./data unless the COURSE_DATA environment variable names another folder
DATA_ROOT = os.environ.get("COURSE_DATA", "./data")
print(f"Data folder: {DATA_ROOT}")
print("Setup complete.")

---
## 1 — Load CIFAR-10 Dataset

CIFAR-10 consists of 60,000 colour images (32x32 pixels, 3 channels) across 10 classes:
airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

### 1.1 Define Transforms, Download Data and Hold Out a Validation Set

The images are independent photographs, so a random validation split is acceptable here
(for patches cut from the same well or image it would not be: see Lab 2B).

In [ ]:
# Basic transforms: convert to tensor, then standardize each channel with the per-channel
# mean and std of the 50,000 CIFAR-10 training images (widely published values). They include
# the images we hold out for validation below; leaving those out changes only the fourth decimal.
transform_basic = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

# Download CIFAR-10
full_train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=True, transform=transform_basic
)
test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_ROOT, train=False, download=True, transform=transform_basic
)

# Class names, in label order (label 0 = airplane, ..., label 9 = truck)
classes = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')
assert list(classes) == full_train_dataset.classes

# Hold out 5,000 training images for validation; the test set is kept for Section 8
train_set, val_set = random_split(
    full_train_dataset, [45000, 5000], generator=torch.Generator().manual_seed(RANDOM_STATE)
)

# Classroom budget: a fixed random subset of the training split (45_000 = use all of it)
N_TRAIN = 20_000
subset_indices = torch.randperm(len(train_set), generator=torch.Generator().manual_seed(RANDOM_STATE))[:N_TRAIN]
train_subset = Subset(train_set, subset_indices.tolist())

print(f"Training samples   : {len(train_subset)} (of {len(train_set)} in the training split)")
print(f"Validation samples : {len(val_set)}")
print(f"Test samples       : {len(test_dataset)} (used once, in Section 8)")
print(f"Image shape        : {full_train_dataset[0][0].shape}")
print(f"Label -> class     : {dict(enumerate(classes))}")

### 1.2 Visualize Sample Images

In [ ]:
# Helper to unnormalize images for display
def imshow(img):
    """Unnormalize an image tensor and return it as an H x W x C array for plt.imshow."""
    img = img * torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1) + \
          torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    img = img.clamp(0, 1)
    npimg = img.numpy()
    return np.transpose(npimg, (1, 2, 0))

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    image, label = train_subset[i]
    ax.imshow(imshow(image))
    ax.set_title(classes[label], fontsize=9)
    ax.axis("off")

plt.suptitle("CIFAR-10 Sample Images (training subset)", fontsize=14)
plt.tight_layout()
plt.show()

### 1.3 Create DataLoaders

In [ ]:
BATCH_SIZE = 64

# num_workers=0 keeps data loading in the notebook process (worker processes can hang in notebooks)
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
# The test loader is created in Section 8, once the model choice is final

# Verify batch shape
images, labels = next(iter(train_loader))
print(f"Batch image shape : {images.shape}")
print(f"Batch label shape : {labels.shape}")

---
## 2 — Build a Simple CNN

Our **SimpleCNN** has two convolutional blocks followed by two fully connected layers:

| Block | Layers |
|-------|--------|
| Conv Block 1 | Conv2d(3, 32, 3, padding=1) + ReLU + MaxPool2d(2) |
| Conv Block 2 | Conv2d(32, 64, 3, padding=1) + ReLU + MaxPool2d(2) |
| Classifier | Flatten + Linear(64*8*8, 512) + ReLU + Linear(512, 10) |

Before training we pass a dummy batch through the model and print the output shape of every
layer: the shape ledger from the slides.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Conv Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Conv Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))   # 16x16 -> 8x8
        x = x.view(x.size(0), -1)              # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

# Shape check: pass a dummy batch through the model and print the output shape of every
# convolution, pooling and linear layer (hooks run in forward order; pool is used twice)
def shape_hook(name):
    return lambda module, inputs, output: print(f"{name:<6} {tuple(output.shape)}")

hooks = [layer.register_forward_hook(shape_hook(name)) for name, layer in model.named_children()
         if isinstance(layer, (nn.Conv2d, nn.MaxPool2d, nn.Linear))]
model.eval()   # evaluation mode for the check; train_one_epoch switches back to training mode
with torch.no_grad():   # a batch of zeros: only the shapes matter, and the seeded random stream is untouched
    model(torch.zeros(8, 3, 32, 32, device=device))
for hook in hooks:
    hook.remove()

---
## 3 — Training Loop

### 3.1 Define Loss and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(f"Loss     : {criterion}")
print(f"Optimizer: Adam, lr=1e-3")

### 3.2 Training and Validation Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch and return average loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate the model and return average loss and accuracy."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total

print("Training and evaluation functions defined.")

### 3.3 Train for 10 Epochs

After every epoch we evaluate on the **validation** set and keep a copy of the weights from
the epoch with the highest validation accuracy.

In [ ]:
NUM_EPOCHS = 10   # classroom budget; with more time, raise it together with N_TRAIN

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc, best_epoch, best_state = 0.0, 0, None

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    if val_acc > best_val_acc:
        best_val_acc, best_epoch = val_acc, epoch
        best_state = copy.deepcopy(model.state_dict())

    print(f"Epoch {epoch:>2d}/{NUM_EPOCHS}  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

model.load_state_dict(best_state)
print(f"\nKeeping the weights from epoch {best_epoch} (validation accuracy {best_val_acc:.4f})")

---
## 4 — Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs, history["train_loss"], "o-", label="Train Loss", color="steelblue")
ax1.plot(epochs, history["val_loss"], "s--", label="Validation Loss", color="tomato")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()

ax2.plot(epochs, history["train_acc"], "o-", label="Train Accuracy", color="steelblue")
ax2.plot(epochs, history["val_acc"], "s--", label="Validation Accuracy", color="tomato")
ax2.axvline(best_epoch, color="gray", linestyle=":", label=f"Kept epoch ({best_epoch})")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Curves")
ax2.legend()

plt.suptitle("SimpleCNN Training on CIFAR-10", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5 — Evaluate on the Validation Set

Error analysis is part of model development, so it uses validation images. The test set is
kept for one final check (Section 8).

### 5.1 Overall Validation Accuracy

In [ ]:
val_loss, val_acc = evaluate(model, val_loader, criterion, device)
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {val_acc:.4f}  (weights from epoch {best_epoch})")

### 5.2 Per-Class Accuracy (validation)

In [ ]:
class_correct = [0] * 10
class_total = [0] * 10

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        for i in range(labels.size(0)):
            label = labels[i].item()
            class_correct[label] += (predicted[i] == label).item()
            class_total[label] += 1

print(f"{'Class':<12s} {'Correct':>8s} {'Total':>6s} {'Accuracy':>9s}")
print("-" * 38)
for i in range(10):
    acc = class_correct[i] / class_total[i] if class_total[i] > 0 else 0
    print(f"{classes[i]:<12s} {class_correct[i]:>8d} {class_total[i]:>6d} {acc:>9.4f}")

---
## 6 — Visualize Misclassified Validation Images

In [ ]:
# Collect eight misclassified validation examples
misclassified = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        mask = predicted != labels
        for idx in mask.nonzero(as_tuple=True)[0]:
            misclassified.append((
                images[idx].cpu(),
                labels[idx].item(),
                predicted[idx].item(),
            ))
        if len(misclassified) >= 8:
            break

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < len(misclassified):
        img, true_label, pred_label = misclassified[i]
        ax.imshow(imshow(img))
        ax.set_title(f"T:{classes[true_label]}\nP:{classes[pred_label]}", fontsize=10)
    ax.axis("off")

plt.suptitle("Misclassified Validation Images (T=True, P=Predicted)", fontsize=13)
plt.tight_layout()
plt.show()

---
## 7 — Improved CNN with BatchNorm and Dropout

We change **one thing**: we add **Batch Normalization** after each convolution (stabilizes
training) and **Dropout** before the final layer (reduces overfitting of the large dense
head). Everything else stays as in SimpleCNN: layers and sizes, optimizer, training images,
epoch budget and seed. A difference on the validation set can then be attributed to this
change.

### 7.1 Define Improved Architecture

In [ ]:
class ImprovedCNN(nn.Module):
    """SimpleCNN plus BatchNorm after each convolution and Dropout before the output layer."""

    def __init__(self):
        super(ImprovedCNN, self).__init__()
        # Conv Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Conv Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)

        # Same classifier sizes as SimpleCNN
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # 32x32 -> 16x16
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # 16x16 -> 8x8
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

torch.manual_seed(RANDOM_STATE)
model_v2 = ImprovedCNN().to(device)
print(model_v2)

total_params_v2 = sum(p.numel() for p in model_v2.parameters())
print(f"\nTotal parameters: {total_params_v2:,} (SimpleCNN: {total_params:,})")

### 7.2 Train the Improved Model

In [ ]:
criterion_v2 = nn.CrossEntropyLoss()
optimizer_v2 = optim.Adam(model_v2.parameters(), lr=1e-3)

# Same budget as SimpleCNN: same training images, same number of epochs
history_v2 = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc_v2, best_epoch_v2, best_state_v2 = 0.0, 0, None

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model_v2, train_loader, criterion_v2, optimizer_v2, device)
    val_loss, val_acc = evaluate(model_v2, val_loader, criterion_v2, device)

    history_v2["train_loss"].append(train_loss)
    history_v2["train_acc"].append(train_acc)
    history_v2["val_loss"].append(val_loss)
    history_v2["val_acc"].append(val_acc)
    if val_acc > best_val_acc_v2:
        best_val_acc_v2, best_epoch_v2 = val_acc, epoch
        best_state_v2 = copy.deepcopy(model_v2.state_dict())

    print(f"Epoch {epoch:>2d}/{NUM_EPOCHS}  "
          f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  "
          f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}")

model_v2.load_state_dict(best_state_v2)
print(f"\nKeeping the weights from epoch {best_epoch_v2} (validation accuracy {best_val_acc_v2:.4f})")

### 7.3 Compare Validation Curves and Choose a Model

We choose the model with the higher best validation accuracy. Differences of a few tenths of
a percent are within run-to-run noise for a single seed; in that case prefer the simpler model.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs, history["val_acc"], "s--", label="SimpleCNN Validation", color="tomato")
ax1.plot(epochs, history_v2["val_acc"], "o-", label="ImprovedCNN Validation", color="steelblue")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Validation Accuracy")
ax1.set_title("Validation Accuracy Comparison")
ax1.legend()

ax2.plot(epochs, history["val_loss"], "s--", label="SimpleCNN Validation", color="tomato")
ax2.plot(epochs, history_v2["val_loss"], "o-", label="ImprovedCNN Validation", color="steelblue")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Validation Loss")
ax2.set_title("Validation Loss Comparison")
ax2.legend()

plt.suptitle("SimpleCNN vs ImprovedCNN on CIFAR-10 (validation set)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"SimpleCNN   best validation acc: {best_val_acc:.4f} (epoch {best_epoch})")
print(f"ImprovedCNN best validation acc: {best_val_acc_v2:.4f} (epoch {best_epoch_v2})")

if best_val_acc_v2 > best_val_acc:
    final_model, final_name = model_v2, "ImprovedCNN"
else:
    final_model, final_name = model, "SimpleCNN"
print(f"Chosen on validation data: {final_name}")

---
## 8 — Final Test Evaluation (once)

The choice is now locked: the model with the higher validation accuracy, with the weights of
its best validation epoch. We evaluate it once on the 10,000 test images. This number
describes the locked procedure; changing the model after seeing it would turn the test set
into another validation set.

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

test_loss, test_acc = evaluate(final_model, test_loader, criterion, device)
print(f"Final model         : {final_name}")
print(f"Final Test Loss     : {test_loss:.4f}")
print(f"Final Test Accuracy : {test_acc:.4f}")

---
## Summary

In this exercise we:

1. **Loaded and explored** the CIFAR-10 dataset (32x32 colour images, 10 classes) and held out
   5,000 training images for validation.
2. **Built a SimpleCNN** with two convolutional blocks and two fully connected layers, and
   printed every layer's output shape for a dummy batch.
3. **Trained** the model and tracked training/validation loss and accuracy, keeping the best
   validation epoch.
4. **Evaluated** per-class accuracy and visualized misclassified images on the validation set.
5. **Changed one thing**, adding BatchNorm and Dropout, and trained with the same budget.
6. **Compared** the two models on validation data, chose one, and evaluated it once on the
   test set.

Key takeaways:
- Model and epoch selection use validation data; the test set is touched once, at the end.
- Change one thing per comparison and give both models the same budget.
- Batch normalization often speeds up convergence, and dropout can reduce overfitting of a
  large dense head; check both claims on validation data rather than assuming them.